In [13]:
import pandas as pd
import os
from sqlalchemy import create_engine
import logging
import time

# Create logs folder if it doesn't exist
os.makedirs("logs", exist_ok=True)

logging.basicConfig(
    filename="logs/ingestion.db.log",
    level=logging.DEBUG,
    format="%(asctime)s-%(levelname)s-%(message)s",
    filemode="a"
)

engine = create_engine('sqlite:///inventory.db')


def ingest_db(df, table_name, engine):
    '''this function will ingest the dataframe into database table'''
    
    df.to_sql(
        table_name,
        con=engine,
        if_exists='replace',
        index=False,
        chunksize=50000
    )


def load_raw_data():
    '''this function will load the CSVs as dataframe and ingest into db'''
    
    start = time.time()

    path = r'C:\Users\sanju\OneDrive\Desktop\Project\data.ipynb\data'

    for file in os.listdir(path):
        
        if file.endswith('.csv'):
            
            file_path = os.path.join(path, file)

            df = pd.read_csv(file_path)

            logging.info(f'Ingesting {file} in db')

            ingest_db(df, file[:-4], engine)

            end = time.time()

            total_time = (end - start) / 60

            logging.info('Ingestion Complete')
            logging.info(f'Total Time Taken: {total_time} minutes')


if __name__ == '__main__':
    load_raw_data()

In [6]:
import logging
import sqlite3
import pandas as pd
from ingestion_db import ingest_db

logging.basicConfig(
    filename="logs/get_vendor_summary.log",
    level=logging.DEBUG,
    format="%(asctime)s - %(levelname)s - %(message)s",
    filemode="a"
)
def create_vendor_summary(conn):
    '''this function will merge the different tables to get the overall vendor summary and adding new columns in the resultant data'''
    vendor_sales_summary=pd.read_sql_query("""WITH FreightSummary AS(
    SELECT
    VendorNumber,
    SUM(Freight)AS FreightCost
    FROM vendor_invoice
    GROUP BY VendorNumber
    ),
    

PurchaseSummary AS (
    SELECT
        p.VendorNumber,
        p.VendorName,
        p.Brand,
        p.Description,
        p.PurchasePrice,
        pp.price AS ActualPrice,
        pp.Volume,
        SUM(p.Quantity)AS TotalPurchaseQuantity,
        SUM(p.Dollars)AS TotalPurchaseDollars
    FROM purchases p
    JOIN purchase_prices pp
        ON p.Brand=pp.Brand
    WHERE p.PurchasePrice>0
    GROUP BY p.VendorNumber,p.VendorName,p.Brand,p.Description,p.PurchasePrice,pp.price,pp.Volume
    ),
    SalesSummary as(
        SELECT
            Vendorno,
            Brand,
            SUM(s.SalesQuantity)AS TotalSalesQuantity,
            SUM(s.SalesDollars)AS TotalSalesDollars,
            SUM(s.SalesPrice)AS TotalSalesPrice,
            SUM(s.ExciseTax)AS TotalExciseTax
        FROM sales s
        GROUP BY VendorNo,Brand
        )
    
        SELECT
            ps.VendorNumber,
            ps.VendorName,
            ps.Brand,
            ps.Description,
            ps.PurchasePrice,
            ps.ActualPrice,
            ps.Volume,
            ps.TotalPurchaseQuantity,
            ps.TotalPurchaseDollars,
            ss.TotalSalesQuantity,
            ss.TotalSalesDollars,
            ss.TotalSalesPrice,
            ss.TotalExciseTax,
            fs.FreightCost
        FROM PurchaseSummary ps
        LEFT JOIN SalesSummary ss
            ON ps.VendorNumber=ss.VendorNo
            AND ps.Brand=ss.Brand
        LEFT JOIN FreightSummary fs
            ON ps.VendorNumber=fs.VendorNumber
        ORDER BY ps.TotalPurchaseDollars DESC""",conn)

    return vendor_sales_summary

def clean_data(df):
    '''this function will clean the data'''
    #changing datatype to float
    df['Volume']=df['Volume'].astype('float')

    #filling missing value with 0
    df.fillna(0,inplace=True)

    #removing spaces from categorical columns
    df['VendorName']=df['VendorName'].str.strip()
    df['Description']=df['Description'].str.strip()

    #creating new columns for better analysis
    vendor_sales_summary['GrossProfit']=vendor_sales_summary['TotalSalesDollars'] - vendor_sales_summary['TotalPurchaseDollars']
    vendor_sales_summary['ProfitMargin']=(vendor_sales_summary['GrossProfit']/vendor_sales_summary['TotalSalesDollars'])*100
    vendor_sales_summary['StockTurnover']=vendor_sales_summary['TotalSalesQuantity']/vendor_sales-summary['TotalPurchaseQuantity']
    vendor_sales_summary['SalesToPurchaseRatio']=vendor_sales_summary['TotalSalesDollars'/vendor_sales_summary['TotalPurchaseDollars']]

    return df

if __name__=='__main__':
    #creating databse Connection
    conn=sqlite3.connect('inventory.dbdb')

    logging.info('Creating Vendor Summary Table.....')
    summary_df=create_vendor_summary(conn)
    logging.info(summary_df.head())

    logging.info('Cleaning Data....')
    clean_df=clean_data(summary_df)
    logging.info(clean_df.head())

    logging.info('Ingesting data.....')
    ingest_db(clean_df,'vendor_sales_summary',conn)
    logging.info('Completed')

DatabaseError: Execution failed on sql 'WITH FreightSummary AS(
    SELECT
    VendorNumber,
    SUM(Freight)AS FreightCost
    FROM vendor_invoice
    GROUP BY VendorNumber
    ),


PurchaseSummary AS (
    SELECT
        p.VendorNumber,
        p.VendorName,
        p.Brand,
        p.Description,
        p.PurchasePrice,
        pp.price AS ActualPrice,
        pp.Volume,
        SUM(p.Quantity)AS TotalPurchaseQuantity,
        SUM(p.Dollars)AS TotalPurchaseDollars
    FROM purchases p
    JOIN purchase_prices pp
        ON p.Brand=pp.Brand
    WHERE p.PurchasePrice>0
    GROUP BY p.VendorNumber,p.VendorName,p.Brand,p.Description,p.PurchasePrice,pp.price,pp.Volume
    ),
    SalesSummary as(
        SELECT
            Vendorno,
            Brand,
            SUM(s.SalesQuantity)AS TotalSalesQuantity,
            SUM(s.SalesDollars)AS TotalSalesDollars,
            SUM(s.SalesPrice)AS TotalSalesPrice,
            SUM(s.ExciseTax)AS TotalExciseTax
        FROM sales s
        GROUP BY

In [7]:
import sqlite3

conn = sqlite3.connect('inventory.db')

tables = conn.execute("""
SELECT name 
FROM sqlite_master 
WHERE type='table'
""").fetchall()

print(tables)

[('begin_inventory',), ('end_inventory',), ('purchases',), ('purchase_prices',), ('sales',), ('vendor_invoice',), ('vendor_sales_summary',)]


In [8]:
import logging
import sqlite3
import pandas as pd

from ingestion_db import ingest_db


# --------------------------------------------------
# Logging Configuration
# --------------------------------------------------

logging.basicConfig(
    filename="logs/get_vendor_summary.log",
    level=logging.DEBUG,
    format="%(asctime)s - %(levelname)s - %(message)s",
    filemode="a"
)


# --------------------------------------------------
# Create Vendor Summary
# --------------------------------------------------

def create_vendor_summary(conn):

    """
    Merge different tables to create
    overall vendor sales summary.
    """

    vendor_sales_summary = pd.read_sql_query(
        """
        WITH FreightSummary AS (

            SELECT
                VendorNumber,
                SUM(Freight) AS FreightCost
            FROM vendor_invoice
            GROUP BY VendorNumber
        ),

        PurchaseSummary AS (

            SELECT
                p.VendorNumber,
                p.VendorName,
                p.Brand,
                p.Description,
                p.PurchasePrice,
                pp.Price AS ActualPrice,
                pp.Volume,

                SUM(p.Quantity) AS TotalPurchaseQuantity,
                SUM(p.Dollars) AS TotalPurchaseDollars

            FROM purchases p

            JOIN purchase_prices pp
                ON p.Brand = pp.Brand

            WHERE p.PurchasePrice > 0

            GROUP BY
                p.VendorNumber,
                p.VendorName,
                p.Brand,
                p.Description,
                p.PurchasePrice,
                pp.Price,
                pp.Volume
        ),

        SalesSummary AS (

            SELECT
                VendorNo,
                Brand,

                SUM(SalesQuantity) AS TotalSalesQuantity,
                SUM(SalesDollars) AS TotalSalesDollars,
                SUM(SalesPrice) AS TotalSalesPrice,
                SUM(ExciseTax) AS TotalExciseTax

            FROM sales

            GROUP BY
                VendorNo,
                Brand
        )

        SELECT

            ps.VendorNumber,
            ps.VendorName,
            ps.Brand,
            ps.Description,
            ps.PurchasePrice,
            ps.ActualPrice,
            ps.Volume,

            ps.TotalPurchaseQuantity,
            ps.TotalPurchaseDollars,

            ss.TotalSalesQuantity,
            ss.TotalSalesDollars,
            ss.TotalSalesPrice,
            ss.TotalExciseTax,

            fs.FreightCost

        FROM PurchaseSummary ps

        LEFT JOIN SalesSummary ss
            ON ps.VendorNumber = ss.VendorNo
            AND ps.Brand = ss.Brand

        LEFT JOIN FreightSummary fs
            ON ps.VendorNumber = fs.VendorNumber

        ORDER BY
            ps.TotalPurchaseDollars DESC
        """,
        conn
    )

    return vendor_sales_summary


# --------------------------------------------------
# Clean Data
# --------------------------------------------------

def clean_data(df):

    """
    Clean vendor summary data and
    create additional analytical columns.
    """

    # Convert Volume to float
    df["Volume"] = pd.to_numeric(
        df["Volume"],
        errors="coerce"
    )

    # Fill missing values
    df.fillna(0, inplace=True)

    # Remove extra spaces
    df["VendorName"] = df["VendorName"].astype(str).str.strip()

    df["Description"] = df["Description"].astype(str).str.strip()

    # --------------------------------------------------
    # New Analytical Columns
    # --------------------------------------------------

    # Gross Profit
    df["GrossProfit"] = (
        df["TotalSalesDollars"]
        - df["TotalPurchaseDollars"]
    )

    # Profit Margin
    df["ProfitMargin"] = (
        df["GrossProfit"]
        / df["TotalSalesDollars"].replace(0, pd.NA)
    ) * 100

    # Stock Turnover
    df["StockTurnover"] = (
        df["TotalSalesQuantity"]
        / df["TotalPurchaseQuantity"].replace(0, pd.NA)
    )

    # Sales to Purchase Ratio
    df["SalesToPurchaseRatio"] = (
        df["TotalSalesDollars"]
        / df["TotalPurchaseDollars"].replace(0, pd.NA)
    )

    # Replace NaN created by division
    df.fillna(0, inplace=True)

    return df


# --------------------------------------------------
# Main Program
# --------------------------------------------------

if __name__ == "__main__":

    try:

        # Database connection
        conn = sqlite3.connect("inventory.db")

        logging.info("Database connection successful.")

        # Check available tables
        tables = pd.read_sql_query(
            """
            SELECT name
            FROM sqlite_master
            WHERE type='table';
            """,
            conn
        )

        logging.info(
            f"Available tables:\n{tables}"
        )

        # Create Vendor Summary
        logging.info(
            "Creating Vendor Summary Table..."
        )

        summary_df = create_vendor_summary(conn)

        logging.info(
            f"Vendor Summary Shape: {summary_df.shape}"
        )

        logging.info(
            f"\n{summary_df.head()}"
        )

        # Clean Data
        logging.info("Cleaning Data...")

        clean_df = clean_data(summary_df)

        logging.info(
            f"Clean Data Shape: {clean_df.shape}"
        )

        logging.info(
            f"\n{clean_df.head()}"
        )

        # Ingest into Database
        logging.info(
            "Ingesting data into database..."
        )

        ingest_db(
            clean_df,
            "vendor_sales_summary",
            conn
        )

        logging.info(
            "Vendor Summary Table created successfully."
        )

        print(
            "Vendor summary created successfully!"
        )

    except Exception as e:

        logging.error(
            f"Error occurred: {e}",
            exc_info=True
        )

        print(
            f"Error: {e}"
        )

    finally:

        if "conn" in locals():
            conn.close()

        logging.info(
            "Database connection closed."
        )

Vendor summary created successfully!
